# NATIVE YOLO26s → YOLO26n pada Poltrada V3

Eksperimen **10** menggunakan teacher baseline YOLO26s yang sudah dilatih; teacher tidak dilatih ulang.
Notebook mandiri untuk Kaggle, Ultralytics **8.4.155** dan W&B **0.28.1**.

**Urutan:** konfigurasi → persiapan → dataset → definisi metode → training/evaluasi serial per seed.
`RUN_FULL_TRAINING=True`: Run All langsung training setelah verifikasi teacher. Jangan menjalankan notebook 10 dan 11
bersamaan pada GPU yang sama. Data, notebook lama, dan run eksperimen lain tidak diubah.

Ini adaptasi pasangan kapasitas teacher–student, bukan klaim reproduksi identik paper CrossKD.
Belum ada hasil GPU/artifact nyata tersimpan dalam notebook yang baru dibuat ini.
**Multi-seed serial:** `SEEDS_TO_RUN=[123, 2024]`; seed 42 sudah dijalankan sebelumnya.
Run All membuat satu run W&B per seed, lengkap dengan training 100 epoch, evaluasi,
upload, penutupan run dan pembersihan GPU sebelum seed berikutnya. Untuk tahap final gunakan
`[7, 3407]`, atau `[123]` untuk menjalankan satu seed. Setelah gagal, jalankan hanya seed
yang belum selesai; tidak ada auto-resume atau pengulangan seed 42.


## 1. Konfigurasi penelitian terpusat

Nilai training mengikuti notebook sumber. Ubah konfigurasi secara eksplisit dan dokumentasikan setiap perubahan; jangan menurunkan batch otomatis.

In [ ]:
# Ubah parameter di bagian ini saja untuk menjalankan eksperimen.
EXPERIMENT_ID = "10_native_yolo26n_teacher_yolo26s_dis1"
METHOD = "native"
DATA = "/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml"  # Dataset bersih di Kaggle.
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
# Seed 42 sudah selesai. Final: [7, 3407]; satu run saja: [123].
SEEDS_TO_RUN = [123, 2024]
SEED = SEEDS_TO_RUN[0] if SEEDS_TO_RUN else None  # Diatur ulang otomatis per run.
DEVICE = 0                   # GPU Kaggle; 'cpu' untuk diagnosis lokal
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True

WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"           # test hanya jika dataset punya split test
PROJECT = "/kaggle/working/results"  # Colab: ganti ke /content/results
KD_WEIGHT = 1.0            # dis: official Ultralytics KD

# W&B online: setiap epoch mengunggah last.pt + best.pt sampai epoch tersebut.
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = "ilhamrafiqin-brawijaya-university"

# Run All langsung menjalankan training 100 epoch, sesuai permintaan pengguna.
RUN_FULL_TRAINING = True
TEACHER_RUN_ID = "q5kp9k9d"
TEACHER_ARTIFACT = "ilhamrafiqin-brawijaya-university/yolom26-kd-indotraffic/02_baseline_yolo26s-q5kp9k9d-checkpoints:v62"
WANDB_RUN_NAME = f"native_T26s_S26n_dis{KD_WEIGHT:g}_seed{SEED}"
REFERENCE_SHA256 = {'02_baseline_yolo26s.ipynb': '871bf920e413375154e3fbf5181f82dc30ae239c460ed9cbc6fb4f096d2cdb11', '05_native_yolo26n.ipynb': '249cc326c7305ce5cac6069db7e91b5d87336c6e171ba76b1601a2486821962b', '07_crosskd_yolo26n.ipynb': '11ac22c97ec7fb90e58dfe6dfd0f6bad61dc597fdccf034049788574e7d59345'}

# Group membandingkan pasangan yang sama; tag mengikuti konfigurasi run baru.
WANDB_GROUP = "T26s_S26n"
WANDB_TAGS = [
    METHOD, 'teacher-s', 'student-n', f"dis{KD_WEIGHT:g}", f"seed{SEED}", "poltrada-v3", EXPERIMENT_ID
]

BASE_WANDB_TAGS = [tag for tag in WANDB_TAGS if not tag.startswith("seed")]


## 2. Persiapan lingkungan dan autentikasi

Aktifkan satu GPU dan Internet Kaggle. Tambahkan Secret `WANDB_API_KEY`. PyTorch/CUDA bawaan Kaggle dipertahankan; library eksperimen dipatok versinya. Run W&B yang sudah aktif tidak ditutup paksa.

In [ ]:
import importlib.metadata as metadata
import os
import subprocess
import sys

print("[ENV] Memeriksa versi paket eksperimen...", flush=True)
for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"[ENV] Menginstal {package}=={version}; menunggu pip selesai...", flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

print("[ENV] Memuat PyTorch, Ultralytics dan W&B...", flush=True)
import base64
import gc
import hashlib
import json
import math
import time
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterator

import torch
from torch import nn
from torch.nn import functional as F
import wandb
from ultralytics import YOLO, settings
from ultralytics.cfg import DEFAULT_CFG_DICT
from ultralytics.data.utils import check_det_dataset
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.distill_model import DistillationModel, FeatureHook
from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.torch_utils import get_flops, init_seeds, strip_optimizer, unwrap_model

settings.update({"wandb": False})
if not torch.cuda.is_available():
    raise RuntimeError("Training penelitian ini memerlukan GPU CUDA Kaggle.")
if DEVICE != 0:
    raise ValueError("Protokol ini menggunakan satu GPU, DEVICE=0; perubahan wajib didokumentasikan.")
torch.cuda.set_device(DEVICE)
if not WANDB_ENABLED:
    raise ValueError("Eksperimen ini memerlukan W&B online untuk provenance dan artifact setiap epoch.")
if not os.environ.get("WANDB_API_KEY"):
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("[W&B] Login menggunakan WANDB_API_KEY dari environment/Secret Kaggle...", flush=True)
wandb.login(key=os.environ["WANDB_API_KEY"])
print("[W&B] Login selesai; menyiapkan sesi baru...", flush=True)
if wandb.run is not None:
    raise RuntimeError("Ada run W&B aktif. Selesaikan sesi itu terlebih dahulu; notebook tidak akan menghentikannya.")
ENVIRONMENT_INFO = {
    "python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda,
    "ultralytics": metadata.version("ultralytics"), "wandb": metadata.version("wandb"),
    "gpu": torch.cuda.get_device_name(DEVICE),
}
print(json.dumps(ENVIRONMENT_INFO, indent=2))


## 3. Memuat dataset dan label

Loader resmi memeriksa Poltrada V3 yang dipasang sebagai input. Hanya validation dipakai untuk pemilihan checkpoint; notebook tidak membersihkan atau mengubah file dataset.

In [ ]:
def class_names(names: dict | list) -> list[str]:
    """Kembalikan label dalam urutan indeks kelas, tanpa mengurutkan nama alfabetis."""
    return ([str(names[key]) for key in sorted(names, key=lambda key: int(key))]
            if isinstance(names, dict) else [str(name) for name in names])


if EVAL_SPLIT != "val":
    raise ValueError("Gunakan val untuk pemilihan checkpoint; test tidak boleh dipakai untuk tuning.")
if not Path(DATA).is_file():
    raise FileNotFoundError(f"Tambahkan dataset Poltrada V3 di Kaggle: {DATA}")
print("[DATA] Memeriksa path dan label dataset...", flush=True)
data_info = check_det_dataset(DATA, autodownload=False)
if not data_info.get("train") or not data_info.get("val") or int(data_info["nc"]) < 1:
    raise ValueError("Dataset harus memiliki train, val, dan kelas deteksi yang valid.")
DATASET_NAMES = class_names(data_info["names"])
print("Dataset:", DATA, "| kelas:", DATASET_NAMES)

## 4. Definisi verifikasi teacher YOLO26s

Setiap run seed menautkan artifact baseline `q5kp9k9d:v62` melalui `run.use_artifact()`.
Pemeriksaan manifest, checksum/SHA-256, producer, schema tensor, dataset, urutan kelas dan
best global tetap dijalankan sebelum training. Versi artifact bukan nomor epoch.
Run W&B dibuka oleh antrean di section 7; cell ini hanya mendefinisikan fungsi verifikasi.


In [ ]:
def log_status(stage: str, message: str) -> None:
    """Tampilkan progres segera, termasuk saat menunggu jaringan atau setup trainer."""
    stamp = datetime.now().astimezone().strftime("%H:%M:%S")
    print(f"[{stamp}] [{stage}] {message}", flush=True)


def file_hash(path: Path, algorithm: str = "sha256") -> str:
    """Hash file secara streaming agar checkpoint tidak diduplikasi di RAM."""
    digest = hashlib.new(algorithm)
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def global_best_evidence(selected: Any, producer: Any) -> dict[str, Any]:
    """Bandingkan best.pt v62 dengan best-so-far terakhir dari run yang selesai."""
    collection = TEACHER_ARTIFACT.rsplit("/", 1)[-1].split(":")[0]
    rows = []
    latest_candidate, latest_epoch = None, -1
    started = time.perf_counter()
    log_status("AUDIT", "Membaca metadata riwayat checkpoint teacher dari W&B...")
    for candidate in producer.logged_artifacts():
        if candidate.type != "model" or candidate.name.split(":")[0] != collection:
            continue
        meta = dict(candidate.metadata or {})
        rows.append({"name": candidate.name, "version": candidate.version,
                     "epoch": meta.get("epoch"), "best_fitness": meta.get("best_fitness"),
                     "best_epoch": meta.get("best_epoch_so_far"),
                     "best_digest": None})
        if isinstance(meta.get("epoch"), int) and meta["epoch"] > latest_epoch:
            latest_candidate, latest_epoch = candidate, meta["epoch"]
        if len(rows) == 1 or len(rows) % 10 == 0:
            log_status("AUDIT", f"Metadata {len(rows)} checkpoint terbaca ({time.perf_counter() - started:.1f}s).")
    valid = [row for row in rows if isinstance(row["epoch"], int)
             and isinstance(row["best_fitness"], (int, float))
             and math.isfinite(row["best_fitness"])]
    evidence = {"verified": False, "source_state": producer.state,
                "source_summary_best_epoch": dict(producer.summary).get("best_epoch"),
                "checkpoint_artifacts": rows, "reason": "metadata belum lengkap"}
    if not valid or len(valid) != len(rows):
        log_status("AUDIT", "Metadata riwayat tidak lengkap; best global belum terverifikasi.")
        return evidence
    latest = max(valid, key=lambda row: row["epoch"])
    # Intermediate manifests are unnecessary: metadata proves the fitness maximum,
    # and only the final best-so-far digest must match the selected checkpoint.
    log_status("AUDIT", f"Memeriksa manifest best checkpoint terakhir, epoch {latest_epoch}...")
    latest_entry = latest_candidate.manifest.entries.get("best.pt")
    latest["best_digest"] = getattr(latest_entry, "digest", None)
    maximum = max(row["best_fitness"] for row in valid)
    epochs = {row["epoch"] for row in valid}
    selected_meta = dict(selected.metadata or {})
    best_entry = selected.manifest.entries.get("best.pt")
    selected_fitness = selected_meta.get("best_fitness")
    summary_epoch = evidence["source_summary_best_epoch"]
    complete = epochs == set(range(1, latest["epoch"] + 1))
    same_score = (isinstance(selected_fitness, (int, float))
                  and math.isclose(float(selected_fitness), maximum, rel_tol=1e-9, abs_tol=1e-12))
    same_digest = bool(best_entry and best_entry.digest == latest["best_digest"])
    same_epoch = (selected_meta.get("best_epoch_so_far") == latest["best_epoch"]
                  and latest["best_epoch"] == summary_epoch and isinstance(summary_epoch, int))
    verified = producer.state == "finished" and complete and same_score and same_digest and same_epoch
    evidence.update(verified=verified, latest_checkpoint_epoch=latest["epoch"],
                    actual_global_best_epoch=latest["best_epoch"], global_best_fitness=maximum,
                    same_final_best_digest=same_digest, same_global_best_score=same_score,
                    artifact_history_complete=complete,
                    reason="best.pt cocok dengan best global run selesai" if verified
                    else "v62 belum terbukti identik dengan best global final; full training diblokir")
    log_status("AUDIT", f"Selesai: {len(rows)} checkpoint, best global verified={verified}, {time.perf_counter() - started:.1f}s.")
    return evidence


def verify_yolo26s_schema(model: nn.Module) -> None:
    """Periksa bentuk seluruh tensor terhadap arsitektur s resmi, bukan label YAML saja."""
    reference = DetectionModel("yolo26s.yaml", nc=int(data_info["nc"]),
                               ch=int(data_info.get("channels", 3)), verbose=False)
    expected = {name: tuple(value.shape) for name, value in reference.state_dict().items()}
    actual = {name: tuple(value.shape) for name, value in model.state_dict().items()}
    del reference
    if expected != actual:
        raise ValueError("Struktur tensor checkpoint tidak cocok dengan arsitektur YOLO26s resmi.")


def verify_teacher_artifact(artifact: Any) -> tuple[str, dict[str, Any]]:
    """Tolak sumber/arsitektur/label/checksum yang tidak cocok; tidak ada teacher fallback."""
    if artifact.version != "v62" or artifact.name.split(":")[0] != "02_baseline_yolo26s-q5kp9k9d-checkpoints":
        raise ValueError("Artifact teacher harus koleksi baseline YOLO26s q5kp9k9d, versi v62.")
    producer = artifact.logged_by()
    if producer is None or producer.id != TEACHER_RUN_ID or list(producer.path) != [WANDB_ENTITY, WANDB_PROJECT, TEACHER_RUN_ID]:
        raise ValueError("Producer artifact bukan run baseline YOLO26s yang ditetapkan.")
    if "best.pt" not in artifact.manifest.entries:
        raise ValueError("Manifest artifact tidak memiliki best.pt; cache lokal tidak boleh menggantikan sumber.")
    artifact_meta = dict(artifact.metadata or {})
    if not isinstance(artifact_meta.get("epoch"), int) or not isinstance(artifact_meta.get("best_epoch_so_far"), int):
        raise ValueError("Metadata epoch artifact/best epoch tidak tersedia; v62 bukan nomor epoch.")
    if artifact_meta.get("experiment") != "02_baseline_yolo26s" or not artifact.digest:
        raise ValueError("Identitas eksperimen atau digest artifact tidak lengkap.")
    entry = artifact.manifest.entries["best.pt"]
    log_status("TEACHER", f"Mengunduh artifact {artifact.name}; tunggu jaringan W&B...")
    artifact_dir = Path(artifact.download(root=str(SESSION_DIR / "teacher_yolo26s_v62")))
    log_status("TEACHER", "Unduhan selesai; memeriksa ukuran dan checksum best.pt...")
    checkpoint = artifact_dir / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(f"best.pt tidak tersedia: {checkpoint}")
    if entry.size is None or checkpoint.stat().st_size != entry.size:
        raise ValueError("Ukuran checkpoint berbeda dari manifest W&B.")
    try:
        expected_md5 = base64.b64decode(entry.digest, validate=True)
    except (ValueError, TypeError) as error:
        raise ValueError("Digest manifest best.pt tidak valid.") from error
    if len(expected_md5) != 16 or bytes.fromhex(file_hash(checkpoint, "md5")) != expected_md5:
        raise ValueError("Checksum checkpoint tidak cocok dengan manifest W&B.")
    sha256 = file_hash(checkpoint)
    log_status("TEACHER", "Memuat checkpoint teacher YOLO26s...")
    teacher = YOLO(str(checkpoint))
    architecture = teacher.model.yaml
    family = str(architecture.get("yaml_file", "")).replace("\\", "/").rsplit("/", 1)[-1].split(".")[0]
    if family not in {"yolo26", "yolo26s"} or architecture.get("scale") != "s":
        raise ValueError("Arsitektur aktual checkpoint bukan YOLO26s.")
    log_status("TEACHER", "Memeriksa schema arsitektur dan urutan kelas teacher...")
    verify_yolo26s_schema(teacher.model)
    if int(teacher.model.nc) != int(data_info["nc"]) or class_names(teacher.model.names) != DATASET_NAMES:
        raise ValueError("Jumlah atau urutan kelas teacher berbeda dari dataset.")
    config = dict(producer.config or {})
    checkpoint_args = dict((teacher.ckpt or {}).get("train_args") or {})
    declared = {}
    for name, values in (("artifact", artifact_meta), ("producer", config), ("checkpoint", checkpoint_args)):
        dataset = values.get("dataset", values.get("data"))
        model = values.get("model")
        if dataset and str(dataset).replace("\\", "/") != DATA.replace("\\", "/"):
            raise ValueError(f"Deklarasi dataset teacher berbeda pada {name}: {dataset}")
        if model and str(model).replace("\\", "/").rsplit("/", 1)[-1] != "yolo26s.pt":
            raise ValueError(f"Deklarasi arsitektur teacher berbeda pada {name}: {model}")
        declared[name] = {"dataset": dataset, "model": model}
    if not config.get("dataset") or not config.get("model") or not checkpoint_args.get("data"):
        raise ValueError("Deklarasi provenance sumber/dataset checkpoint tidak lengkap.")
    teacher.model.eval().requires_grad_(False)
    if teacher.model.training or any(parameter.requires_grad for parameter in teacher.model.parameters()):
        raise RuntimeError("Teacher gagal dibekukan.")
    global_best = global_best_evidence(artifact, producer)
    provenance = {
        "requested_artifact": TEACHER_ARTIFACT, "resolved_name": artifact.name,
        "version": artifact.version, "artifact_digest": artifact.digest,
        "manifest_digest": entry.digest, "best_pt_sha256": sha256, "checkpoint_path": str(checkpoint),
        "producer_id": producer.id, "producer_path": list(producer.path), "producer_state": producer.state,
        "artifact_epoch": artifact_meta["epoch"], "actual_best_epoch": artifact_meta["best_epoch_so_far"],
        "artifact_metadata": artifact_meta, "declared_sources": declared,
        "checkpoint_epoch_zero_based": (teacher.ckpt or {}).get("epoch"),
        "architecture": "yolo26s", "architecture_tensor_schema_verified": True,
        "nc": int(teacher.model.nc), "names": DATASET_NAMES,
        "global_best": global_best, "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    del teacher
    return str(checkpoint), provenance



def prepare_seed_teacher():
    global TEACHER_CHECKPOINT, TEACHER_PROVENANCE, teacher_path
    log_status("TEACHER", f"Mengakses artifact teacher {TEACHER_ARTIFACT} untuk seed {SEED}...")
    teacher_artifact = wb_run.use_artifact(TEACHER_ARTIFACT, type="model")
    TEACHER_CHECKPOINT, TEACHER_PROVENANCE = verify_teacher_artifact(teacher_artifact)
    (SESSION_DIR / "teacher_provenance.json").write_text(json.dumps(TEACHER_PROVENANCE, indent=2), encoding="utf-8")
    wb_run.config.update({"teacher_provenance": TEACHER_PROVENANCE})
    if not TEACHER_PROVENANCE["global_best"]["verified"]:
        raise RuntimeError("best.pt v62 belum terverifikasi sebagai best global final dari source run.")
    teacher_path = Path(TEACHER_CHECKPOINT)
    KD_TRAIN_ARGS["distill_model"] = str(teacher_path)
    log_status("TEACHER", f"Teacher terverifikasi: SHA256={TEACHER_PROVENANCE['best_pt_sha256']}.")


## 5. Metode distillation dan pengaman protokol

Native KD menggunakan `DistillationModel` resmi dengan koefisien `dis=1.0` sesuai konfigurasi penelitian terbaru.
CrossKD menyalin objective notebook 07: quality focal classification, confidence-weighted GIoU box,
projector channel dan statistik fitur, serta teacher-head suffix differentiable terhadap fitur student.
Forward target teacher memakai `no_grad`; suffix cross tetap dilalui autograd meskipun bobot teacher frozen.
Tidak ada CSAKD, AT, PKD, atau objective tambahan.

Pengaman berikut hanya menyaring parameter frozen dari optimizer, mengunci batch/AMP, dan
menonaktifkan retry OOM yang dapat mengubah batch pada Ultralytics 8.4.155. Checkpoint publik
berisi student-only; teacher dan adapter KD tidak menambah parameter inferensi student.

In [ ]:
def audit_optimizer(model: nn.Module, optimizer: torch.optim.Optimizer) -> dict[str, int]:
    """Pastikan parameter trainable tercakup tepat sekali dan teacher tidak masuk optimizer."""
    trainable = {id(parameter) for parameter in model.parameters() if parameter.requires_grad}
    actual = [id(parameter) for group in optimizer.param_groups for parameter in group["params"]]
    if len(actual) != len(set(actual)) or set(actual) != trainable:
        raise RuntimeError("Optimizer tidak mencakup seluruh parameter trainable tepat sekali.")
    teacher = getattr(model, "teacher_model", None)
    if teacher is None or teacher.training or any(parameter.requires_grad for parameter in teacher.parameters()):
        raise RuntimeError("Teacher harus frozen dan eval selama training.")
    if set(actual) & {id(parameter) for parameter in teacher.parameters()}:
        raise RuntimeError("Parameter teacher tidak boleh berada dalam optimizer.")
    return {"trainable_parameter_tensors": len(trainable), "optimizer_parameter_tensors": len(actual)}


def strict_batch_start(trainer: Any) -> None:
    """Nonaktifkan recovery OOM otomatis sebelum forward/backward pada versi yang dipatok."""
    trainer._oom_retries = 3  # Guard engine 8.4.155: langsung raise, tidak mengubah batch.
    if trainer.batch_size != BATCH or trainer.args.batch != BATCH or trainer.args.imgsz != IMGSZ:
        raise RuntimeError("Batch atau resolusi efektif berbeda dari protokol penelitian.")
    if bool(trainer.amp) != AMP:
        raise RuntimeError("AMP efektif berubah; perlu keputusan pengguna sebelum melanjutkan.")
    net = unwrap_model(trainer.model)
    if net.teacher_model.training or any(parameter.requires_grad for parameter in net.teacher_model.parameters()):
        raise RuntimeError("Teacher tidak frozen/eval.")


def strict_batch_end(trainer: Any) -> None:
    """Pulihkan counter hanya sesudah batch berhasil; bukan setelah OOM."""
    trainer._oom_retries = 0
    if not torch.isfinite(trainer.loss).all():
        raise RuntimeError("Detection/KD objective non-finite; hentikan eksperimen.")
    if any(parameter.grad is not None for parameter in unwrap_model(trainer.model).teacher_model.parameters()):
        raise RuntimeError("Ditemukan gradien pada teacher.")


class ProtocolTrainerMixin:
    """Kontrol optimizer/protokol; tidak mengganti objective Native KD atau CrossKD."""
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        if self.world_size > 1 or self.args.compile or self.args.resume:
            raise ValueError("Gunakan satu GPU, compile=False, resume=False pada eksperimen ini.")
        self.add_callback("on_train_batch_start", strict_batch_start)
        self.add_callback("on_train_batch_end", strict_batch_end)

    def _build_train_pipeline(self) -> None:
        if self.batch_size != BATCH or self.args.batch != BATCH:
            raise RuntimeError("OOM/batch berubah. Batch tidak akan diturunkan tanpa keputusan pengguna.")
        super()._build_train_pipeline()

    def build_optimizer(self, model: nn.Module, name: str = "auto", lr: float = 0.001,
                        momentum: float = 0.9, decay: float = 1e-5, iterations: float = 1e5) -> Any:
        optimizer = super().build_optimizer(model, name, lr, momentum, decay, iterations)
        for group in optimizer.param_groups:
            group["params"] = [parameter for parameter in group["params"] if parameter.requires_grad]
        audit_optimizer(model, optimizer)
        return optimizer

    def _setup_train(self) -> None:
        super()._setup_train()
        if self.device.type != "cuda" or self.world_size > 1:
            raise RuntimeError("Training memerlukan satu GPU CUDA.")
        strict_batch_start(self)
        self._oom_retries = 0
        audit_optimizer(unwrap_model(self.model), self.optimizer)

    def train(self) -> Any:
        try:
            return super().train()
        finally:
            net = unwrap_model(self.model)
            if hasattr(net, "_remove_feature_hooks"):
                net._remove_feature_hooks()


@contextmanager
def exclusive_gpu() -> Iterator[None]:
    """Cegah notebook Native/CrossKD pada workspace yang sama memakai GPU bersamaan."""
    lock = Path(PROJECT).parent / ".yolo26kd_teacher_s_gpu.lock"
    lock.parent.mkdir(parents=True, exist_ok=True)
    try:
        descriptor = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
    except FileExistsError as error:
        raise RuntimeError(f"GPU dikunci eksperimen lain: {lock}. Periksa proses aktif sebelum menghapus lock usang.") from error
    try:
        with os.fdopen(descriptor, "w") as stream:
            stream.write(json.dumps({"pid": os.getpid(), "experiment": EXPERIMENT_ID, "run": wb_run.id}))
        yield
    finally:
        lock.unlink(missing_ok=True)


def export_student_checkpoint(source_path: Path, destination: Path) -> str:
    """Ekspor salinan student-only tanpa memodifikasi checkpoint training asli."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source_path.resolve() == destination.resolve():
        raise ValueError("Ekspor student harus menggunakan lokasi berbeda.")
    checkpoint = strip_optimizer(source_path, str(destination), updates={"teacher_provenance": TEACHER_PROVENANCE})
    if not checkpoint or not destination.is_file():
        raise RuntimeError("Ekspor checkpoint student gagal.")
    net = checkpoint["model"]
    if isinstance(net, DistillationModel) or hasattr(net, "teacher_model"):
        raise RuntimeError("Checkpoint inferensi masih berisi wrapper/teacher.")
    if net.yaml.get("scale") != "n":
        raise RuntimeError("Checkpoint deployment harus student YOLO26n.")
    del checkpoint
    return file_hash(destination)


def protocol_args(name: str, plots: bool = True) -> dict[str, Any]:
    """Parameter sumber tetap; default augmentasi berasal dari Ultralytics 8.4.155."""
    args = dict(data=DATA, model=MODEL_WEIGHTS, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
                weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
                project=str(SESSION_DIR), name=name, exist_ok=False,
                workers=WORKERS, patience=PATIENCE, plots=plots, amp=AMP, pretrained=True,
                save=True, compile=False, resume=False, nbs=DEFAULT_CFG_DICT["nbs"])
    args.update(KD_TRAIN_ARGS)
    return args

In [ ]:
class ResearchTrainer(ProtocolTrainerMixin, DetectionTrainer):
    """Integrasi Native KD resmi; loss dibangun oleh engine Ultralytics tanpa override."""
    pass

KD_TRAIN_ARGS = {"distill_model": None, "dis": KD_WEIGHT}
KD_CONFIG = {"dis": KD_WEIGHT, "implementation": "ultralytics.nn.distill_model.DistillationModel", "loss_changed": False}


## 6. Definisi training 100 epoch dan antrean serial

`SEEDS_TO_RUN=[123, 2024]`; seed 42 yang sudah selesai tidak dijalankan ulang.
Cell berikut menyiapkan fungsi; antrean mulai pada cell eksekusi di section 7.
Setiap seed memulai student baru, RNG baru, run W&B baru (`resume="never"`), state callback
dan output terpisah. Training, evaluasi, upload sinkron dan finish selesai sebelum seed berikutnya.
Jika terjadi kegagalan, antrean dihentikan; hasil seed sebelumnya tetap tersimpan.
Rumus loss, optimizer step/GradScaler, batch, resolusi, AMP dan protokol training tetap sama.


In [ ]:
wb_state = {"saved_epochs": set(), "uploaded_epochs": set(), "logged_epochs": set(), "best_epoch": None,
            "epoch_upload_seconds": []}


def save_epoch_artifact(trainer: Any) -> None:
    """Salin student-only ke snapshot epoch unik; tunggu upload sebelum file dapat berubah."""
    epoch = int(trainer.epoch) + 1
    fitness, best_fitness = float(trainer.fitness), float(trainer.best_fitness)
    if not math.isfinite(fitness) or not math.isfinite(best_fitness):
        raise RuntimeError("Fitness tidak finite; checkpoint tidak boleh diklaim valid.")
    is_best = fitness == best_fitness
    if is_best:
        wb_state["best_epoch"] = epoch
    wb_state["saved_epochs"].add(epoch)
    log_epoch_metrics(trainer)
    log_status("EPOCH", f"Epoch {epoch}/{trainer.epochs} selesai; metrik dikirim. Menyiapkan checkpoint student-only...")
    snapshot = Path(trainer.save_dir) / "epoch_snapshots" / f"epoch-{epoch:03d}"
    snapshot.mkdir(parents=True, exist_ok=False)
    last_sha = export_student_checkpoint(Path(trainer.last), snapshot / "last.pt")
    best_sha = export_student_checkpoint(Path(trainer.best), snapshot / "best.pt")
    artifact = wandb.Artifact(name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints", type="model",
        metadata={"experiment": EXPERIMENT_ID, "seed": SEED, "epoch": epoch, "fitness": fitness,
                  "best_fitness": best_fitness, "is_best_epoch": is_best,
                  "best_epoch_so_far": wb_state["best_epoch"], "teacher_provenance": TEACHER_PROVENANCE,
                  "last_pt_sha256": last_sha, "best_pt_sha256": best_sha,
                  "student_only": True, "method": METHOD})
    for name in ("last.pt", "best.pt"):
        artifact.add_file(str(snapshot / name), name=name)
    for name in ("results.csv", "args.yaml"):
        path = Path(trainer.save_dir) / name
        if path.is_file():
            artifact.add_file(str(path), name=name)
    artifact.add_file(str(SESSION_DIR / "teacher_provenance.json"), name="teacher_provenance.json")
    aliases = [f"epoch-{epoch:03d}", "latest"] + (["best"] if is_best else [])
    start = time.perf_counter()
    log_status("W&B", f"Mengunggah checkpoint epoch {epoch}; epoch berikutnya menunggu upload selesai...")
    wb_run.log_artifact(artifact, aliases=aliases).wait()
    wb_state["epoch_upload_seconds"].append(time.perf_counter() - start)
    wb_state["uploaded_epochs"].add(epoch)
    log_status("W&B", f"Upload checkpoint epoch {epoch} selesai ({wb_state['epoch_upload_seconds'][-1]:.1f}s).")


def log_epoch_metrics(trainer: Any) -> None:
    """Satu record per epoch termasuk detection/KD loss, validation, LR, dan best epoch."""
    epoch = int(trainer.epoch) + 1
    if epoch not in wb_state["saved_epochs"] or epoch in wb_state["logged_epochs"]:
        return
    values = {"epoch": epoch, **{key: float(value) for key, value in trainer.label_loss_items(trainer.tloss, prefix="train").items()},
              **{key: float(value) for key, value in trainer.metrics.items()},
              **{key: float(value) for key, value in trainer.lr.items()},
              "checkpoint/best_epoch_so_far": wb_state["best_epoch"],
              "protocol/physical_batch": trainer.batch_size, "protocol/accumulate": trainer.accumulate,
              "protocol/effective_batch": trainer.batch_size * trainer.accumulate,
              "amp/grad_scale": float(trainer.scaler.get_scale()),
              "memory/peak_allocated_gib": torch.cuda.max_memory_allocated(DEVICE) / 1024**3}
    wb_run.log(values, step=epoch)
    wb_state["logged_epochs"].add(epoch)


def require_full_training() -> None:
    """Periksa CUDA, teacher, dan flag training; tidak mensyaratkan smoke test."""
    if not RUN_FULL_TRAINING:
        raise RuntimeError("RUN_FULL_TRAINING masih False.")
    if not torch.cuda.is_available():
        raise RuntimeError("Training final memerlukan GPU CUDA.")
    if not TEACHER_PROVENANCE["global_best"]["verified"]:
        raise RuntimeError("best.pt v62 belum terverifikasi sebagai best global final dari source run.")
    if file_hash(Path(TEACHER_CHECKPOINT)) != TEACHER_PROVENANCE["best_pt_sha256"]:
        raise RuntimeError("SHA256 teacher berubah sejak verifikasi.")


def run_full_training() -> dict[str, Any]:
    """Jalankan protokol 100 epoch dari pretrained student dengan AMP engine resmi."""
    require_full_training()
    student = None
    with exclusive_gpu():
        try:
            init_seeds(SEED, deterministic=True)
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(DEVICE)
            log_status("TRAIN", f"Memuat student {MODEL_WEIGHTS}...")
            student = YOLO(MODEL_WEIGHTS)
            student.add_callback("on_model_save", bind_seed_callback(save_epoch_artifact))
            student.add_callback("on_fit_epoch_end", bind_seed_callback(log_epoch_metrics))
            student.add_callback("on_train_start", lambda trainer: log_status("TRAIN", "Setup selesai; memulai epoch pertama."))
            arguments = protocol_args("training")
            arguments.pop("model")  # objek YOLO sudah menentukan model student.
            wb_run.config.update({"training_args": arguments, "kd": KD_CONFIG,
                                  "reference_sha256": REFERENCE_SHA256,
                                  "training_mode": "direct_100_epochs", "smoke_required": False,
                                  "run_full_training": True})
            start = time.perf_counter()
            log_status("TRAIN", f"Memulai setup training {EPOCHS} epoch: batch={BATCH}, imgsz={IMGSZ}, amp={AMP}; loader, AMP check dan projector KD disiapkan...")
            student.train(trainer=ResearchTrainer, **arguments)
            duration = time.perf_counter() - start
            directory = Path(student.trainer.save_dir)
            if not (directory / "weights/best.pt").is_file():
                raise FileNotFoundError("Training selesai tanpa best.pt.")
            return {"run_dir": str(directory), "best_pt": str(directory / "weights/best.pt"),
                    "last_pt": str(directory / "weights/last.pt"), "training_seconds": duration,
                    "completed_epochs": len(wb_state["uploaded_epochs"]), "best_epoch": wb_state["best_epoch"],
                    "peak_allocated_gib": torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
                    "peak_reserved_gib": torch.cuda.max_memory_reserved(DEVICE) / 1024**3,
                    "artifact_upload_seconds": sum(wb_state["epoch_upload_seconds"])}
        finally:
            student = None
            gc.collect()
            torch.cuda.empty_cache()


In [ ]:
def validate_seed_list(seeds):
    """Reject ambiguous queues before opening any run or creating a model."""
    if not isinstance(seeds, (list, tuple)) or not seeds:
        raise ValueError("SEEDS_TO_RUN harus berupa daftar seed yang tidak kosong.")
    if any(isinstance(seed, bool) or not isinstance(seed, int) or not 0 <= seed < 2**32 for seed in seeds):
        raise ValueError("Setiap seed harus integer dalam rentang 0 sampai 2**32-1.")
    if len(seeds) != len(set(seeds)):
        raise ValueError("SEEDS_TO_RUN tidak boleh memiliki seed duplikat.")
    return list(seeds)


def seed_run_name(seed):
    if METHOD == "none":
        return f"baseline_yolo26n_seed{seed}"
    if METHOD == "native":
        return f"native_T26s_S26n_dis{KD_WEIGHT:g}_seed{seed}"
    return f"crosskd_T26s_S26n_seed{seed}"


def bind_seed_callback(callback):
    """Bind the active seed/run/state; reject stale callbacks instead of mixing logs."""
    seed, run, state = SEED, wb_run, wb_state
    def bound(trainer):
        if SEED != seed or wb_run is not run or wb_state is not state:
            raise RuntimeError("Callback berasal dari seed/run lain; logging dihentikan.")
        return callback(trainer)
    return bound


def start_seed_run():
    global wb_run
    NEW_WANDB_RUN_ID = wandb.util.generate_id()
    log_status("W&B", f"Membuat run baru: name={WANDB_RUN_NAME}, id={NEW_WANDB_RUN_ID}, resume=never...")
    diagnostic = DATA == "coco8.yaml"
    wb_run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=WANDB_RUN_NAME,
        id=NEW_WANDB_RUN_ID, resume="never", settings=wandb.Settings(init_timeout=90),
        group=f"diagnostic-{WANDB_GROUP}" if diagnostic else WANDB_GROUP, job_type="training",
        tags=[*WANDB_TAGS, "diagnostic" if diagnostic else "research"],
        config={"experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
                "dataset": DATA, "eval_split": EVAL_SPLIT, "seed": SEED,
                "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH, "device": str(DEVICE),
                "workers": WORKERS, "optimizer": OPTIMIZER, "learning_rate": LR,
                "patience": PATIENCE, "amp": AMP, "weight_decay": WEIGHT_DECAY,
                "warmup_epochs": WARMUP_EPOCHS, "close_mosaic": CLOSE_MOSAIC,
                "teacher_artifact": TEACHER_ARTIFACT if METHOD != "none" else None,
                "environment": ENVIRONMENT_INFO, "seeds_to_run": list(SEEDS_TO_RUN)})
    log_status("W&B", f"Run aktif: {wb_run.url}")


def run_one_seed(seed, queue_dir):
    """Complete one independent run, including evaluation/upload/finish, before returning."""
    global SEED, SESSION_DIR, WANDB_RUN_NAME, WANDB_TAGS, wb_run, wb_state
    global TEACHER_CHECKPOINT, TEACHER_PROVENANCE, teacher_path
    if wandb.run is not None:
        raise RuntimeError("Masih ada run W&B aktif; seed baru tidak boleh dimulai.")
    SEED = seed
    WANDB_RUN_NAME = seed_run_name(seed)
    WANDB_TAGS = [*BASE_WANDB_TAGS, f"seed{seed}"]
    SESSION_DIR = Path(queue_dir) / f"seed-{seed}"
    SESSION_DIR.mkdir(parents=True, exist_ok=False)
    wb_run = None
    wb_state = {"saved_epochs": set(), "uploaded_epochs": set(), "logged_epochs": set(),
                "best_epoch": None, "epoch_upload_seconds": []}
    if METHOD != "none":
        TEACHER_CHECKPOINT, TEACHER_PROVENANCE, teacher_path = None, {}, None
        KD_TRAIN_ARGS["distill_model"] = None
    try:
        init_seeds(seed, deterministic=True)
        start_seed_run()
        prepare_seed_teacher()
        log_status("SEED", f"Memulai seed {seed}: {EPOCHS} epoch, student baru dari {MODEL_WEIGHTS}.")
        training = run_full_training()
        (SESSION_DIR / "training_result.json").write_text(json.dumps(training, indent=2), encoding="utf-8")
        with exclusive_gpu():
            result = evaluate_final_training(training)
        result.update(seed=seed, wandb_run_id=wb_run.id, wandb_run_name=WANDB_RUN_NAME,
                      wandb_url=wb_run.url, seed_directory=str(SESSION_DIR))
        (SESSION_DIR / "seed_result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
        # finish() waits for the run's remaining W&B sync; the next seed starts afterwards.
        wb_run.finish()
        log_status("SEED", f"Seed {seed} selesai; run W&B sudah ditutup.")
        return result
    except BaseException:
        if wb_run is not None:
            try:
                wb_run.finish(exit_code=1)
            except Exception as finish_error:
                log_status("W&B", f"Penutupan run gagal ({type(finish_error).__name__}); antrean dihentikan.")
        raise
    finally:
        wb_run = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_seed_queue():
    """Serial queue: persist completed seeds and stop immediately on any failure."""
    seeds = validate_seed_list(SEEDS_TO_RUN)
    if not WANDB_ENABLED:
        raise ValueError("Antrean penelitian ini memerlukan W&B online.")
    if wandb.run is not None:
        raise RuntimeError("Selesaikan run W&B aktif sebelum menjalankan antrean ini.")
    token = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    queue_dir = Path(PROJECT) / EXPERIMENT_ID / f"multiseed_{token}"
    queue_dir.mkdir(parents=True, exist_ok=False)
    results = []
    progress = {"experiment": EXPERIMENT_ID, "seeds_to_run": seeds, "completed": [],
                "failed_seed": None, "status": "running"}
    progress_path = queue_dir / "multiseed_progress.json"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Seed serial {seeds}; hasil: {queue_dir}. Seed 42 tidak ditambahkan otomatis.")
    for seed in seeds:
        try:
            result = run_one_seed(seed, queue_dir)
            results.append(result)
            (queue_dir / "multiseed_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
            progress["completed"].append({"seed": seed, "wandb_run_id": result["wandb_run_id"],
                                          "seed_directory": result["seed_directory"]})
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
        except BaseException:
            progress.update(failed_seed=seed, status="failed")
            progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
            log_status("QUEUE", f"Berhenti pada seed {seed}. Hasil seed selesai tetap tersimpan; untuk lanjut isi SEEDS_TO_RUN dengan seed yang belum selesai.")
            raise
    progress["status"] = "completed"
    progress_path.write_text(json.dumps(progress, indent=2), encoding="utf-8")
    log_status("QUEUE", f"Selesai: {len(results)} run terpisah. Gabungkan hasil 42 yang sudah ada saat pelaporan.")
    return results

# Eksekusi antrean di section 7, sesudah fungsi evaluasi didefinisikan.


## 7. Evaluasi student terbaik dan pelaporan

Evaluasi hanya berjalan sesudah training final. Laporkan mAP50–95, mAP50, precision, recall,
parameter/GFLOPs **student-only**, latency inferensi validation dengan protokol baseline,
durasi training (termasuk checkpoint/upload), peak VRAM, best epoch, dan provenance teacher.
Nilai yang tidak tersedia disimpan `null`, bukan hasil perkiraan. Test split tidak dipakai.

In [ ]:
def evaluate_final_training(training: dict[str, Any]) -> dict[str, Any]:
    """Evaluate/upload one seed and always release its inference model."""
    best = metrics = None
    try:
        best = YOLO(training["best_pt"])
        if isinstance(best.model, DistillationModel) or best.model.yaml.get("scale") != "n":
            raise RuntimeError("Model inferensi harus student YOLO26n tanpa teacher/adapter KD.")
        metrics = best.val(data=DATA, split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                           plots=True, verbose=False, project=str(SESSION_DIR), name="final_validation")
        parameters = sum(parameter.numel() for parameter in best.model.parameters())
        try:
            gflops = float(get_flops(best.model, imgsz=IMGSZ))
        except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
            print("GFLOPs belum tersedia:", error)
            gflops = None
        results = {**training, "experiment": EXPERIMENT_ID, "method": METHOD, "diagnostic_only": False,
                   "dataset": DATA, "eval_split": "val", "seed": SEED, "imgsz": IMGSZ, "batch": BATCH,
                   "map50_95": float(metrics.box.map), "map50": float(metrics.box.map50),
                   "precision": float(metrics.box.mp), "recall": float(metrics.box.mr),
                   "parameters_student": int(parameters), "gflops_student": gflops,
                   "inference_ms_per_image": float(metrics.speed["inference"]),
                   "inference_protocol": {"split": "val", "batch": BATCH, "imgsz": IMGSZ,
                                          "precision": "Ultralytics default, sama baseline", "speed": metrics.speed},
                   "teacher_provenance": TEACHER_PROVENANCE, "environment": ENVIRONMENT_INFO}
        path = SESSION_DIR / "metrics.json"
        path.write_text(json.dumps(results, indent=2), encoding="utf-8")
        config_path = SESSION_DIR / "config.json"
        config_path.write_text(json.dumps({"args": protocol_args("training"), "kd": KD_CONFIG,
                                         "reference_sha256": REFERENCE_SHA256}, indent=2), encoding="utf-8")
        wb_run.summary.update({f"final/{key}": value for key, value in results.items()
                               if isinstance(value, (int, float, str)) and key != "teacher_provenance"})
        artifact = wandb.Artifact(f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
                                  metadata={"teacher_provenance": TEACHER_PROVENANCE, "seed": SEED, "best_epoch": training["best_epoch"]})
        for item in (path, config_path, SESSION_DIR / "teacher_provenance.json"):
            artifact.add_file(str(item), name=item.name)
        for folder in (Path(training["run_dir"]), Path(metrics.save_dir)):
            for plot in sorted(folder.glob("*.png")):
                key = f"plots/{folder.name}/{plot.stem}"
                wb_run.log({key: wandb.Image(str(plot))})
                artifact.add_file(str(plot), name=f"{folder.name}/{plot.name}")
        wb_run.log_artifact(artifact, aliases=["final"]).wait()
        return results
    finally:
        best = metrics = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
MULTISEED_RESULTS = []
if RUN_FULL_TRAINING:
    MULTISEED_RESULTS = run_seed_queue()
    print(json.dumps(MULTISEED_RESULTS, indent=2))
else:
    print("Training tidak dijalankan karena RUN_FULL_TRAINING=False; tidak ada run W&B dibuka.")


## 8. Hasil multi-seed dan penutupan

Setiap seed menyimpan `training_result.json`, `metrics.json`, `config.json`, `seed_result.json`
dan checkpoint/artifact-nya sendiri. Direktori antrean memuat `multiseed_results.json` serta
`multiseed_progress.json`. Tidak ada perhitungan gabungan yang menganggap seed 42 sudah diimpor.
Gabungkan run 42 yang sudah ada saat melaporkan mean ± standar deviasi.

Untuk tahap final gunakan `SEEDS_TO_RUN=[7, 3407]`; untuk satu run saja gunakan `[123]`.
Setelah kegagalan, pilih seed yang belum selesai sebelum Run All kembali.
Pengujian CPU/mock tidak membuktikan stabilitas atau durasi training GPU Kaggle.


In [ ]:
print("Hasil antrean:", len(MULTISEED_RESULTS), "seed selesai. Setiap run sudah ditutup.")
